# Cleaned Data of ZüriWieNeu Reports

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

In [2]:
list(DATA_RAW.iterdir())

[PosixPath('../data/raw/quartiere_polygon.json'),
 PosixPath('../data/raw/quartiere_attributes.json'),
 PosixPath('../data/raw/ZüriWieNeu'),
 PosixPath('../data/raw/Stadtische Quartiere'),
 PosixPath('../data/raw/stzh.adm_statistische_quartiere_map.json'),
 PosixPath('../data/raw/.ipynb_checkpoints'),
 PosixPath('../data/raw/zueriwieneu_meldungen.json')]

In [3]:
reports = gpd.read_file(DATA_RAW / "zueriwieneu_meldungen.json")
quartiere_attr = gpd.read_file(DATA_RAW / "quartiere_attributes.json")
quartiere_poly = gpd.read_file(DATA_RAW / "quartiere_polygon.json")

## Merge Quartiere Attributes with their Polygons

In [4]:
quartiere_attr.head()

,objid,objectid,ori,hali,vali,name,kuerzel,geometry
0,1,1,0,1,2,Affoltern,111,POINT (8.5065 47.42319)
1,2,2,0,1,2,Seebach,119,POINT (8.5396 47.4239)
2,3,3,0,1,2,Saatlen,121,POINT (8.56448 47.41185)
3,4,4,0,1,2,Höngg,101,POINT (8.49567 47.40812)
4,5,5,0,1,2,Wipkingen,102,POINT (8.52337 47.39722)


In [5]:
quartiere_poly.head()

,objid,objectid,geometry
0,1,1,"POLYGON ((8.50583 47.36922, 8.50583 47.36922, ..."
1,10,2,"POLYGON ((8.57525 47.36377, 8.57526 47.36377, ..."
2,11,3,"POLYGON ((8.51547 47.38334, 8.51549 47.38325, ..."
3,12,4,"POLYGON ((8.49837 47.39205, 8.49851 47.39201, ..."
4,13,5,"POLYGON ((8.52281 47.36317, 8.5228 47.363, 8.5..."


In [6]:
quartiere_merge = quartiere_poly.merge(
    quartiere_attr[["objectid", "name", "kuerzel"]],
    on="objectid",
    how="left"
)

quartiere_merge.head()

,objid,objectid,geometry,name,kuerzel
0,1,1,"POLYGON ((8.50583 47.36922, 8.50583 47.36922, ...",Affoltern,111
1,10,2,"POLYGON ((8.57525 47.36377, 8.57526 47.36377, ...",Seebach,119
2,11,3,"POLYGON ((8.51547 47.38334, 8.51549 47.38325, ...",Saatlen,121
3,12,4,"POLYGON ((8.49837 47.39205, 8.49851 47.39201, ...",Höngg,101
4,13,5,"POLYGON ((8.52281 47.36317, 8.5228 47.363, 8.5...",Wipkingen,102


In [18]:
quartiere_merge.to_file(
    DATA_PROCESSED / "quartiere_merge.gpkg"
)

## Spatial Join of Merged Quartiere and the Reports

In [8]:
reports.head()

,objectid,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,userid,title,detail,media_url,interface_used,service_notice,description,url,geometry
0,1,1,20130314151615,20130404072505,20130412075930,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (8.48423 47.37404)
1,2,2,20130314151757,20130326140505,20130412080022,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (8.50819 47.39512)
2,3,4,20130315091416,20130315095505,20130412080810,2684605,1251431,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,https://www.zueriwieneu.ch/photo/4.0.jpeg?bfbb...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Beim Trotto: Beim Trottoir sind einige Randste...,https://www.zueriwieneu.ch/report/4,POINT (8.55959 47.40826)
3,4,5,20130315091715,20130320100505,20130412080905,2681754,1250376,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Par,Auf dem Parkplatz beim Waidspital sind einige ...,https://www.zueriwieneu.ch/photo/5.0.jpeg?e309...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Par: Auf dem Parkplatz beim Waidspital...,https://www.zueriwieneu.ch/report/5,POINT (8.52163 47.39913)
4,5,6,20130315103653,20130422182505,20130423135033,2683094,1247762,Abfall/Sammelstelle,Abfall/Sammelstelle,fixed - council,16624,Arbeitskist,Arbeitskiste ist rund herum verschmiert,https://www.zueriwieneu.ch/photo/6.0.jpeg?8e65...,Web interface,Dieses Graffiti wird von uns in den kommenden ...,Arbeitskist: Arbeitskiste ist rund herum versc...,https://www.zueriwieneu.ch/report/6,POINT (8.53889 47.37545)


In [9]:
# Check if the Coordinate Reference System is the same and the data is within the same bounds
print(reports.crs)
print(quartiere_merge.crs)

print("reports bounds:", reports.total_bounds)
print("quartiere bounds:", quartiere_merge.total_bounds)

EPSG:4326
EPSG:4326
reports bounds: [ 8.4591626  47.32246207  8.62459958 47.43460084]
quartiere bounds: [ 8.44801822 47.32021842  8.62545284 47.43466506]


In [10]:
# conntects each reporting point to its neighborhood polygon
reports_quartiere = gpd.sjoin(
    reports,
    quartiere_merge,
    how="left", #reports are prioritized & every point is kept
    predicate="within" #point needs to be inside polygon
)

reports_quartiere.head()

,objectid_left,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,...,interface_used,service_notice,description,url,geometry,index_right,objid,objectid_right,name,kuerzel
0,1,1,20130314151615,20130404072505,20130412075930,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (8.48423 47.37404),16.0,23,16.0,Fluntern,71
1,2,2,20130314151757,20130326140505,20130412080022,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (8.50819 47.39512),20.0,28,21.0,City,14
2,3,4,20130315091416,20130315095505,20130412080810,2684605,1251431,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Beim Trotto: Beim Trottoir sind einige Randste...,https://www.zueriwieneu.ch/report/4,POINT (8.55959 47.40826),33.0,8,33.0,Hirslanden,73
3,4,5,20130315091715,20130320100505,20130412080905,2681754,1250376,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Par: Auf dem Parkplatz beim Waidspital...,https://www.zueriwieneu.ch/report/5,POINT (8.52163 47.39913),21.0,29,22.0,Lindenhof,13
4,5,6,20130315103653,20130422182505,20130423135033,2683094,1247762,Abfall/Sammelstelle,Abfall/Sammelstelle,fixed - council,...,Web interface,Dieses Graffiti wird von uns in den kommenden ...,Arbeitskist: Arbeitskiste ist rund herum versc...,https://www.zueriwieneu.ch/report/6,POINT (8.53889 47.37545),15.0,25,18.0,Albisrieden,91


In [11]:
print(reports_quartiere.info())
print(reports_quartiere.isna().sum()) # check how much data is left with Na
print(reports_quartiere.duplicated().sum()) # check how many duplicates there are

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 73249 entries, 0 to 73248
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   objectid_left         73249 non-null  int32   
 1   service_request_id    73249 non-null  str     
 2   requested_datetime    73249 non-null  str     
 3   agency_sent_datetime  72417 non-null  str     
 4   updated_datetime      73249 non-null  str     
 5   e                     73249 non-null  int32   
 6   n                     73249 non-null  int32   
 7   service_code          73249 non-null  str     
 8   service_name          73249 non-null  str     
 9   status                73249 non-null  str     
 10  userid                73249 non-null  int32   
 11  title                 73249 non-null  str     
 12  detail                73249 non-null  str     
 13  media_url             73249 non-null  str     
 14  interface_used        73249 non-null  str     

In [12]:
# Create a new DataFrame with only the relevant information for the spatial analysis later
reports_quartiere_clean = reports_quartiere[
    [
        "service_request_id",
        "requested_datetime",
        "updated_datetime",
        "service_code",
        "status",
        "userid",
        "title",
        "detail",
        "geometry",
        "name",
        "kuerzel"
    ]
].copy()

In [13]:
# Rename certain columns using a dictionary to make the DataFrame more readable
reports_quartiere_clean = reports_quartiere_clean.rename(columns={
    "name": "quartier_name",
    "kuerzel": "quartier_code"
})

In [14]:
# make the Date and Time more readable
reports_quartiere_clean["requested_datetime"] = pd.to_datetime(
    reports_quartiere_clean["requested_datetime"]
)

reports_quartiere_clean["updated_datetime"] = pd.to_datetime(
    reports_quartiere_clean["updated_datetime"]
)

reports_quartiere_clean.head()

,service_request_id,requested_datetime,updated_datetime,service_code,status,userid,title,detail,geometry,quartier_name,quartier_code
0,1,2013-03-14 15:16:15,2013-04-12 07:59:30,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,POINT (8.48423 47.37404),Fluntern,71
1,2,2013-03-14 15:17:57,2013-04-12 08:00:22,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,POINT (8.50819 47.39512),City,14
2,4,2013-03-15 09:14:16,2013-04-12 08:08:10,Strasse/Trottoir/Platz,fixed - council,16624,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,POINT (8.55959 47.40826),Hirslanden,73
3,5,2013-03-15 09:17:15,2013-04-12 08:09:05,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Par,Auf dem Parkplatz beim Waidspital sind einige ...,POINT (8.52163 47.39913),Lindenhof,13
4,6,2013-03-15 10:36:53,2013-04-23 13:50:33,Abfall/Sammelstelle,fixed - council,16624,Arbeitskist,Arbeitskiste ist rund herum verschmiert,POINT (8.53889 47.37545),Albisrieden,91


In [15]:
# check which reports didn't get joined to a specific quartier
reports_quartiere_clean[
    reports_quartiere_clean["quartier_name"].isna()
    ]

,service_request_id,requested_datetime,updated_datetime,service_code,status,userid,title,detail,geometry,quartier_name,quartier_code
20262,21901,2019-10-13 18:07:22,2019-10-14 08:18:53,Graffiti,fixed - council,16624,Der Stromka,Der Stromkasten bei der S18 Station Zürich Reh...,POINT (8.58287 47.35079),NaN,NaN
22276,24036,2020-04-13 12:47:47,2020-04-14 15:35:40,Abfall/Sammelstelle,fixed - council,16624,Behälter fü,Behälter für Hundekot und Abfallbehälter überf...,POINT (8.58987 47.39481),NaN,NaN
22414,24180,2020-04-21 15:43:53,2020-04-23 16:27:02,Strasse/Trottoir/Platz,external,16624,An der Tram,An der Tramhaltestelle Fernsehstudio stürzte m...,POINT (8.56171 47.41819),NaN,NaN
51603,57742,2024-06-24 09:42:31,2024-06-24 10:03:10,Signalisation/Lichtsignal,fixed - council,14808,Auto auf de,Auto auf der Tafel verklebt,POINT (8.46591 47.39808),NaN,NaN
57888,64685,2025-01-06 18:35:19,2025-01-07 10:44:28,Signalisation/Lichtsignal,fixed - council,16086,Rote Drücke,Rote Drücker an Fussgänger-Ampel fehlt. Es han...,POINT (8.5564 47.42059),NaN,NaN
64357,71835,2025-07-29 14:42:01,2025-08-04 09:51:34,Beleuchtung/Uhren,fixed - council,16128,Unschönes G,Unschönes Graffiti,POINT (8.51725 47.43139),NaN,NaN


In [16]:
# Drop these reports with quartiere Na
reports_quartiere_clean = reports_quartiere_clean.dropna(
    subset=["quartier_name"]
)

In [17]:
reports_quartiere_clean.to_file(
    DATA_PROCESSED / "reports_quartiere_clean.gpkg"
)